# SMARD German Electricity Market Analysis

Hourly German electricity market data from SMARD (Bundesnetzagentur), 1 Jan 2024 – 31 Dec 2025.

Pipeline: clean -> EDA -> visualizations -> insights -> ML -> Power BI.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import pandas as pd

from src import config, data_loader, features

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## Phase 1 — Clean

Parse the three raw SMARD exports, merge them onto one hourly timestamp index, engineer features, and write `data/interim/clean_hourly.csv` — the only file every later phase reads.

**Acceptance criteria:** 17,544 rows; 2 nulls in `price`; `Nuclear` at 95.96% null; `net_residual_load` correlating 1.000 with SMARD's own `Residual load`.

In [ ]:
raw = data_loader.load_and_merge()
print(raw.shape)
raw.head()

(17544, 17)


,Biomass,Hydropower,Wind offshore,Wind onshore,Photovoltaics,Other renewable,Nuclear,Lignite,Hard coal,Fossil gas,Pumped storage generation,Other conventional,load,Grid load incl. hydro pumped storage,Pumped storage consumption,Residual load,price
timestamp,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,3920.75,1968.50,5679.25,29583.75,3.25,142.0,0.0,3391.75,1860.75,2893.25,489.50,1589.75,40170.25,42193.50,2023.25,4904.00,0.10
2024-01-01 01:00:00,3884.50,1981.75,5345.75,29493.00,3.00,141.5,0.0,3372.75,1854.75,2824.50,415.75,1538.25,38818.25,41342.25,2524.00,3976.50,0.01
2024-01-01 02:00:00,3869.00,2002.00,5188.50,29573.50,3.00,141.0,0.0,3373.50,1828.75,2852.25,403.75,1531.75,37847.75,40899.25,3051.50,3082.75,0.00
2024-01-01 03:00:00,3844.00,2010.50,4674.25,29037.50,3.25,141.0,0.0,3393.00,1817.50,2874.25,325.50,1549.00,37123.25,40216.25,3093.00,3408.25,-0.01
2024-01-01 04:00:00,3849.50,2032.75,4439.25,28970.00,3.00,141.0,0.0,3393.75,1832.75,2857.00,177.75,1515.75,36753.75,40034.00,3280.25,3341.50,-0.03


### Null profile

Profile nulls BEFORE any dropna. A column whose `last_valid` is far from the series end is a structural break, not random missingness.

In [ ]:
nulls = data_loader.null_profile(raw)
nulls

,null_count,null_pct,n_unique,first_valid,last_valid
Nuclear,16836,95.964432,1,2024-01-01,2024-01-30 11:00:00
Biomass,2,0.011400,9978,2024-01-01,2025-12-31 23:00:00
Hydropower,2,0.011400,9491,2024-01-01,2025-12-31 23:00:00
Wind onshore,2,0.011400,16768,2024-01-01,2025-12-31 23:00:00
Wind offshore,2,0.011400,14557,2024-01-01,2025-12-31 23:00:00
Photovoltaics,2,0.011400,10675,2024-01-01,2025-12-31 23:00:00
Other renewable,2,0.011400,3553,2024-01-01,2025-12-31 23:00:00
Lignite,2,0.011400,15005,2024-01-01,2025-12-31 23:00:00
Hard coal,2,0.011400,13691,2024-01-01,2025-12-31 23:00:00
Fossil gas,2,0.011400,14984,2024-01-01,2025-12-31 23:00:00


### Feature engineering

In [ ]:
df = features.add_all(raw)
print(df.shape)
df[["price", "load", "renewable_share", "vre_share",
    "net_residual_load", "price_lag24"]].head()

(17544, 34)


,price,load,renewable_share,vre_share,net_residual_load,price_lag24
timestamp,,,,,,
2024-01-01 00:00:00,0.10,40170.25,0.809231,0.691048,4904.00,NaN
2024-01-01 01:00:00,0.01,38818.25,0.809867,0.690760,3976.50,NaN
2024-01-01 02:00:00,0.00,37847.75,0.809658,0.690285,3082.75,NaN
2024-01-01 03:00:00,-0.01,37123.25,0.804764,0.683261,3408.25,NaN
2024-01-01 04:00:00,-0.03,36753.75,0.804236,0.681399,3341.50,NaN


### Acceptance checks

In [ ]:
n_rows = len(df)
assert n_rows == 17544, f"row count is {n_rows}, expected 17544"
print(f"PASS: row count == {n_rows}")

n_price_nulls = df["price"].isna().sum()
assert n_price_nulls == 2, f"price nulls is {n_price_nulls}, expected 2"
print(f"PASS: price nulls == {n_price_nulls}")

nuclear_null_pct = round(raw["Nuclear"].isna().mean() * 100, 2)
assert nuclear_null_pct == 95.96, f"Nuclear null pct is {nuclear_null_pct}, expected 95.96"
print(f"PASS: Nuclear null pct == {nuclear_null_pct}")

residual_corr = round(df["net_residual_load"].corr(df["Residual load"]), 3)
assert residual_corr == 1.000, f"net_residual_load corr is {residual_corr}, expected 1.000"
print(f"PASS: net_residual_load corr == {residual_corr}")

n_negative = df["is_negative_price"].sum()
assert n_negative == 1030, f"negative-price hours is {n_negative}, expected 1030"
print(f"PASS: negative-price hours == {n_negative}")

PASS: row count == 17544
PASS: price nulls == 2
PASS: Nuclear null pct == 95.96
PASS: net_residual_load corr == 1.0
PASS: negative-price hours == 1030.0


### Save cleaned dataset

This is the only file written in phase 1. Every later phase reads it instead of re-parsing the raw CSVs.

In [ ]:
config.INTERIM.mkdir(parents=True, exist_ok=True)

try:
    df.to_csv(config.CLEAN_FILE)
    print(f"saved -> {config.CLEAN_FILE}")
except PermissionError:
    alt = str(config.CLEAN_FILE).replace(".csv", "_new.csv")
    df.to_csv(alt)
    print(f"[!] file locked (open in Excel?) -> wrote {alt}")

saved -> C:\Users\User\Documents\German Electricity Market Analysis (SMARD)\data\interim\clean_hourly.csv


## Phase 1 complete

_Phase 2 (EDA) comes next._